In [0]:
"""NeuroPlex Batch Ingestion: Genetics (CIViC, cBioPortal, GTR) + Expression (HPA, Monarch) + Other (UniProt, Reactome, KEGG).

Runs all 8 ingestors for the standard neuroscience gene panel.
"""
import sys, time, json
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F

import sys, os
from pathlib import PurePosixPath

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()
p = PurePosixPath(notebook_path)
repo_root = str(p.parent.parent if p.parent.name == "ingestion" else p.parent)
sys.path.insert(0, repo_root)

for mod in list(sys.modules.keys()):
    if "ingestion" in mod:
        del sys.modules[mod]

from ingestion.source_registry import SOURCE_MAP
from ingestion.ingestors.civic import CivicIngestor
from ingestion.ingestors.cbioportal import CbioportalIngestor
from ingestion.ingestors.ncbi_gtr import NcbiGtrIngestor
from ingestion.ingestors.human_protein_atlas import HumanProteinAtlasIngestor
from ingestion.ingestors.monarch import MonarchIngestor
from ingestion.ingestors.uniprot import UniprotIngestor
from ingestion.ingestors.reactome import ReactomeIngestor
from ingestion.ingestors.kegg import KeggIngestor

# ── Configuration ──
TARGET_GENES = [
    "HCRT", "HCRTR1", "HCRTR2",
    "PSEN1", "PSEN2", "APP", "MAPT", "GRN", "TARDBP",
    "SOD1", "FUS", "TBK1",
    "CHD8", "SCN2A", "SYNGAP1",
]

from config.neuroplex_config import load_config
CFG = load_config()
CATALOG = CFG.catalog
SCHEMA = CFG.schema

SCHEMA_DEF = StructType([
    StructField("record_id", StringType(), False),
    StructField("source_key", StringType(), False),
    StructField("gene_symbol", StringType(), True),
    StructField("disease", StringType(), True),
    StructField("drug", StringType(), True),
    StructField("title", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("payload", StringType(), False),
    StructField("ingested_at", StringType(), False),
    StructField("source_updated_at", StringType(), True),
])

def write_to_table(rows, table_name):
    """Write normalized rows to a Delta table."""
    if not rows:
        print(f"  \u26A0\uFE0F {table_name}: no records to write")
        return 0
    fqn = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.createDataFrame(rows, schema=SCHEMA_DEF)
    df = df.withColumn("ingested_at", F.to_timestamp("ingested_at")) \
           .withColumn("source_updated_at", F.to_timestamp("source_updated_at")) \
           .withColumn("payload", F.parse_json("payload"))
    df.write.mode("overwrite").saveAsTable(fqn)
    count = spark.sql(f"SELECT COUNT(*) FROM {fqn}").collect()[0][0]
    print(f"  \u2705 {fqn}: {count} records")
    return count

def run_ingestor(ingestor, source_key, table_name):
    """Run a single ingestor across all target genes."""
    print(f"\n\u2500\u2500 {source_key} \u2500\u2500")
    all_rows = []
    errors = []
    for gene in TARGET_GENES:
        try:
            for raw in ingestor.fetch(gene=gene, limit=30):
                all_rows.append(ingestor.normalize(raw).to_row())
        except Exception as e:
            errors.append((gene, str(e)[:60]))
    print(f"  Fetched: {len(all_rows)} records ({len(errors)} errors)")
    if errors:
        print(f"  Errors: {errors[:3]}")
    return write_to_table(all_rows, table_name)

# ── Run All 8 Ingestors ──
start = time.time()
totals = {}

# Genetics
totals["civic"] = run_ingestor(CivicIngestor(SOURCE_MAP["civic"]), "CIViC", "neuroplex_civic")
totals["cbioportal"] = run_ingestor(CbioportalIngestor(SOURCE_MAP["cbioportal"]), "cBioPortal", "neuroplex_cbioportal")
totals["ncbi_gtr"] = run_ingestor(NcbiGtrIngestor(SOURCE_MAP["ncbi_gtr"]), "NCBI GTR", "neuroplex_ncbi_gtr")

# Expression
totals["hpa"] = run_ingestor(HumanProteinAtlasIngestor(SOURCE_MAP["human_protein_atlas"]), "Human Protein Atlas", "neuroplex_human_protein_atlas")
totals["monarch"] = run_ingestor(MonarchIngestor(SOURCE_MAP["monarch"]), "Monarch Initiative", "neuroplex_monarch")

# Other (Protein + Pathways)
totals["uniprot"] = run_ingestor(UniprotIngestor(SOURCE_MAP["uniprot"]), "UniProt", "neuroplex_uniprot")
totals["reactome"] = run_ingestor(ReactomeIngestor(SOURCE_MAP["reactome"]), "Reactome", "neuroplex_reactome")
totals["kegg"] = run_ingestor(KeggIngestor(SOURCE_MAP["kegg"]), "KEGG", "neuroplex_kegg")

# ── Summary ──
elapsed = time.time() - start
print(f"\n{'\u2550'*50}")
print(f"COMPLETE: {sum(totals.values())} total records across {len(totals)} sources")
print(f"Elapsed: {elapsed:.0f}s")
for src, cnt in totals.items():
    print(f"  {src:20s}: {cnt:>5} records")